In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("store_revenue").getOrCreate()

data = [
    (1, "2023-01-01", 500),
    (1, "2023-01-02", 600),
    (1, "2023-01-03", 450),
    (1, "2023-01-07", 700),
    (1, "2023-01-08", 550),
    (1, "2023-01-09", 800),
    (2, "2023-01-01", 300),
    (2, "2023-01-02", 350),
    (2, "2023-01-06", 400),
    (2, "2023-01-07", 420),
]

columns = ["store_id", "revenue_date", "revenue"]

daily_revenue = spark.createDataFrame(data, columns)


# 1. Cast revenue_date to DateType and ensure revenue is clean
df = daily_revenue.withColumn("revenue_date", F.to_date("revenue_date", "yyyy-MM-dd")) \
                  .withColumn("revenue", F.col("revenue").cast("double"))


In [0]:
# 2. Generate a complete daily calendar for each store to account for implicit zero days
store_bounds = df.groupBy("store_id").agg(
    F.min("revenue_date").alias("min_date"),
    F.max("revenue_date").alias("max_date")
)

# Expand each stores date range into individual daily rows
store_calendar = store_bounds.withColumn(
    "revenue_date",
    F.explode(F.sequence(F.col("min_date"), F.col("max_date"), F.expr("interval 1 day")))
).select("store_id", "revenue_date")

# Join back to original data: missing days become explicit 0.0 revenue
full_daily = store_calendar.join(df, on=["store_id", "revenue_date"], how="left") \
    .withColumn("revenue", F.coalesce(F.col("revenue"), F.lit(0.0)))


In [0]:
# 3. Identify consecutive zero-revenue days using the 'gaps-and-islands' technique
w_store = Window.partitionBy("store_id").orderBy("revenue_date")

# Mark rows that have zero revenue (both explicit and implicit)
flagged = full_daily.withColumn("is_zero", F.when(F.col("revenue") == 0, 1).otherwise(0))

# Create contiguous island groups of zeros
# Date minus (row_number * 1 day) produces a constant anchor date for consecutive days
zero_days = flagged.filter(F.col("is_zero") == 1) \
    .withColumn("rn", F.row_number().over(w_store)) \
    .withColumn("group_id", F.expr("date_sub(revenue_date, rn)"))


In [0]:
# 4. Aggregate groups into gap periods and filter for gaps >= 3 days
gap_periods = zero_days.groupBy("store_id", "group_id").agg(
    F.min("revenue_date").alias("gap_start"),
    F.max("revenue_date").alias("gap_end"),
    F.count("*").alias("gap_days")
).filter(F.col("gap_days") >= 3)


In [0]:
# 5. Calculate total revenue 7 days before gap_start and 7 days after gap_end
# Joining gap_periods with full_daily across the corresponding 7-day ranges
before_rev = gap_periods.alias("g").join(
    full_daily.alias("d"),
    (F.col("g.store_id") == F.col("d.store_id")) &
    (F.col("d.revenue_date") >= F.date_sub(F.col("g.gap_start"), 7)) &
    (F.col("d.revenue_date") < F.col("g.gap_start")),
    how="left"
).groupBy("g.store_id", "g.gap_start", "g.gap_end", "g.gap_days") \
 .agg(F.coalesce(F.sum("d.revenue"), F.lit(0.0)).alias("revenue_before_gap"))

final_df = before_rev.alias("b").join(
    full_daily.alias("d"),
    (F.col("b.store_id") == F.col("d.store_id")) &
    (F.col("d.revenue_date") > F.col("b.gap_end")) &
    (F.col("d.revenue_date") <= F.date_add(F.col("b.gap_end"), 7)),
    how="left"
).groupBy(
    "b.store_id", "b.gap_start", "b.gap_end", "b.gap_days", "b.revenue_before_gap"
).agg(
    F.coalesce(F.sum("d.revenue"), F.lit(0.0)).alias("revenue_after_gap")
)


In [0]:
# 6. Format dates back to string and sort output
result = final_df.select(
    F.col("store_id"),
    F.date_format("gap_start", "yyyy-MM-dd").alias("gap_start"),
    F.date_format("gap_end", "yyyy-MM-dd").alias("gap_end"),
    F.col("gap_days"),
    F.round("revenue_before_gap", 2).alias("revenue_before_gap"),
    F.round("revenue_after_gap", 2).alias("revenue_after_gap")
).orderBy(
    F.col("gap_days").desc(),
    F.col("store_id").asc(),
    F.col("gap_start").asc()
)

result.show()